In [33]:
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [34]:
import torch
import torch.nn as nn

# Q1 - creation of a simple transformer
model = nn.Transformer(d_model=512,
                        nhead=8,
                        num_encoder_layers=6,
                        num_decoder_layers=6)
print(model)

/home/shuptik/pw/sem6/ann/Transformers/.venv/lib/python3.12/site-packages/torch/nn/modules/transformer.py:144: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.encoder = TransformerEncoder(


Transformer(
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-5): 6 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
        )
        (linear1): Linear(in_features=512, out_features=2048, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=2048, out_features=512, bias=True)
        (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
    (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  )
  (decoder): TransformerDecoder(
    (layers): ModuleList(
      (0-5): 6 x TransformerDecoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=512, o

In [35]:
import math
import torch
import torch.nn as nn

class TokenEmbedding(nn.Module):
    def __init__(self, embedding_dim: int, vocab_size: int) -> None:
        super().__init__()
        self.embedding_dim = embedding_dim
        self.vocab_size = vocab_size
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.embedding(x) * math.sqrt(self.embedding_dim) # this is to properly scale token embedding wrt to positional embedding

In [36]:
import math
import torch.nn as nn
import torch

class PositionalEncoding(nn.Module):
    def __init__(self, block_size: int, embedding_dim: int) -> None:
        super().__init__()
        positional_embeddings = torch.zeros(block_size, embedding_dim)
        for pos in range(block_size):
            for index in range(embedding_dim // 2):
                denominator: float = 10000 ** (2 * index / embedding_dim)
                positional_embeddings[pos, 2 * index] = math.sin(pos / denominator)
                positional_embeddings[pos, 2 * index + 1] = math.cos(pos / denominator)
        self.register_buffer("positional_embeddings", positional_embeddings)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.positional_embeddings[:x.size(dim = 1), :]
    
vocab_size = 10000
embedding_dim = 512
sequence_length = 10
batch_size = 2
token_embedding = TokenEmbedding(embedding_dim=embedding_dim, vocab_size=vocab_size)
positional_embedding = PositionalEncoding(block_size=100, embedding_dim=embedding_dim)
id_tensor: torch.Tensor = torch.randint(low = 0, high=10, size = (batch_size, sequence_length))
result = positional_embedding(token_embedding(id_tensor))
print(result.shape)
print(result)

torch.Size([2, 10, 512])
tensor([[[ -5.4018,  -2.9198,  10.1244,  ...,  26.7102,  10.1319,  14.7798],
         [-33.3026,  -7.3607,  17.8557,  ..., -12.9832,  19.1043,  13.5076],
         [-33.2348,  -8.3172,  17.9703,  ..., -12.9832,  19.1045,  13.5076],
         ...,
         [ -7.5666,   2.6039, -13.9417,  ...,  10.0680, -44.3307, -12.7565],
         [ -4.4124,  -4.0653,  11.1151,  ...,  26.7102,  10.1328,  14.7798],
         [ 13.0832, -22.9654,  10.0410,  ..., -20.3934,  25.8550, -19.6649]],

        [[-17.7489, -31.0882,  20.3879,  ...,  -8.5196,  17.7413,  11.5874],
         [ 12.1508,  53.8437,   3.0021,  ..., -18.9839, -19.9532, -17.1185],
         [ -7.3143,   1.4338, -13.4577,  ...,  10.0680, -44.3312, -12.7565],
         ...,
         [-20.2859,  47.4655,  33.9536,  ...,  -3.8420, -23.4220,  -5.6464],
         [ 13.6604, -22.1998,  10.3553,  ..., -20.3934,  25.8549, -19.6649],
         [ -7.8115,   0.9388, -13.7177,  ...,  10.0680, -44.3305, -12.7565]]],
       grad_fn=<Add

In [37]:
import torch.nn.functional as F

class AttentionHead(nn.Module):
    def __init__(self, head_size: int, masked: bool = False) -> None:
        super().__init__()
        self.head_size = head_size
        self.query = nn.Linear(in_features=head_size, out_features=head_size, bias=False)
        self.key = nn.Linear(in_features = head_size, out_features=head_size, bias=False)
        self.values = nn.Linear(in_features = head_size, out_features=head_size, bias=False)
        self.masked = masked
    def forward(self, x: torch.Tensor):
        (B, T, D) = x.shape
        q = self.query(x)
        k = self.key(x)
        v = self.values(x)
        weights =  (q @ k.transpose(-2, -1))*self.head_size ** -0.5 # (B,S,S) shape
        if self.masked:
            tril = torch.tril(torch.ones(T, T, device = x.device))
            weights = weights.masked_fill(tril[:T, :T] == 0, float('-inf'))
        weights = F.softmax(weights, dim = -1)
        scores = weights @ v # (B,S,D)
        return scores


class MultiHeadAttention(nn.Module):
    def __init__(self, n_heads: int, embedding_dim: int, masked = False) -> None:
        super().__init__()
        self.n_heads = n_heads
        self.embedding_dim = embedding_dim
        self.head_dim = embedding_dim // n_heads
        self.attention_heads = nn.ModuleList(
            AttentionHead(self.head_dim, masked)
            for _ in range(n_heads)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        head_outputs = []
        for index in range(self.n_heads):
            start = index * self.head_dim
            end = start + self.head_dim
            x_head = x[:, :, start:end]
            attention = self.attention_heads[index](x_head)
            head_outputs.append(attention)
        return torch.cat(head_outputs, dim=-1)
    
n_heads = 8
embedding_dim = 512
attention_block = MultiHeadAttention(n_heads=n_heads, embedding_dim=embedding_dim)
input = result
result = attention_block(input)
print(result.shape)
print(result.shape == input.shape)


torch.Size([2, 10, 512])
True


In [38]:
import torch.nn as nn
import torch

class FeedForward(nn.Module):
    def __init__(self, embedding_dim : int, dropout: float) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features=embedding_dim, out_features=4*embedding_dim),
            nn.ReLU(),
            nn.Linear(in_features=4 * embedding_dim, out_features=embedding_dim),
            nn.Dropout(dropout)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

class EncoderLayer(nn.Module):
    def __init__(self, n_heads: int, embedding_dim: int, dropout: float) -> None:
        super().__init__()
        self.ln1 = nn.LayerNorm(embedding_dim)
        self.ln2 = nn.LayerNorm(embedding_dim)
        self.fwd = FeedForward(embedding_dim, dropout)
        self.ma = MultiHeadAttention(n_heads=n_heads, embedding_dim=embedding_dim)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.ma(self.ln1(x))
        x = x + self.fwd(self.ln2(x))
        return x

class ClassificationHead(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()
        self.linear = nn.Linear(in_features=in_features, out_features=out_features)
    
    def forward(self, x: torch.Tensor):
        return self.linear(x)
        


class TransformerEncoder(nn.Module):
    def __init__(self, layer_num : int, n_heads: int, embedding_dim: int, vocab_size: int, block_size: int, num_classes: int, dropout: float) -> None:
        super().__init__()
        
        self.embedding = TokenEmbedding(embedding_dim, vocab_size)
        self.pos_embedding = PositionalEncoding(block_size, embedding_dim)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.Sequential(*[EncoderLayer(n_heads, embedding_dim, dropout) for _ in range(layer_num)])
        self.ln = nn.LayerNorm(embedding_dim)
        self.ch = ClassificationHead(in_features=embedding_dim, out_features=num_classes)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        idx = self.embedding(idx)
        idx = self.pos_embedding(idx)
        idx = self.drop(idx)
        idx = self.blocks(idx) 
        idx = self.ln(idx)
        idx = idx.mean(dim=1)  # [B, D]
        logits = self.ch(idx)
        return logits
    
layer_num = 6
n_heads = 8
num_classes = 3
embedding_dim = 512
block_size = 10
vocab_size = 10000
id_tensor: torch.Tensor = torch.randint(low = 0, high=vocab_size, size = (2, block_size - 1))
model = TransformerEncoder(layer_num, n_heads, embedding_dim, vocab_size, block_size, num_classes, dropout=0.4)
logits = model(id_tensor)
probs = F.softmax(logits, dim = -1)
result = probs.argmax(dim = -1)
print(result)
print(result.shape)
print(probs)
print(probs.shape)

tensor([2, 2])
torch.Size([2])
tensor([[0.3403, 0.2423, 0.4174],
        [0.2270, 0.3799, 0.3931]], grad_fn=<SoftmaxBackward0>)
torch.Size([2, 3])


In [42]:
class DecoderLayer(nn.Module):
    def __init__(self, n_heads:int, embedding_dim: int,dropout: float):
        super().__init__()
        self.ln1 = nn.LayerNorm(embedding_dim)
        self.ln2 = nn.LayerNorm(embedding_dim)
        self.ma = MultiHeadAttention(n_heads, embedding_dim, masked = True)
        self.fwd = FeedForward(embedding_dim, dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x =  x + self.ma(self.ln1(x))
        return x + self.fwd(self.ln2(x))
    
class TransformerDecoder(nn.Module):
    def __init__(self, layer_num : int, n_heads: int, embedding_dim: int, vocab_size: int, block_size: int, dropout: float) -> None:
        super().__init__()
        self.embedding = TokenEmbedding(embedding_dim, vocab_size)
        self.pos_embedding = PositionalEncoding(block_size, embedding_dim)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.Sequential(*[DecoderLayer(n_heads, embedding_dim, dropout) for _ in range(layer_num)])
        self.ln = nn.LayerNorm(embedding_dim)
        self.linear = nn.Linear(in_features=embedding_dim, out_features=vocab_size)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        idx = self.embedding(idx)
        idx = self.pos_embedding(idx)
        idx = self.drop(idx)
        idx = self.blocks(idx) 
        idx = self.ln(idx)
        logits = self.linear(idx)
        return logits
    
layer_num = 6
n_heads = 8
num_classes = 3
embedding_dim = 512
block_size = 10
vocab_size = 10000
id_tensor: torch.Tensor = torch.randint(low = 0, high=vocab_size, size = (2, block_size - 1))
model = TransformerDecoder(layer_num, n_heads, embedding_dim, vocab_size, block_size, dropout=0.4)
logits = model(id_tensor)[:, -1, :]
probs = F.softmax(logits, dim=-1)
next_token = torch.multinomial(probs, num_samples=1)
print(logits.shape)
print(next_token)

torch.Size([2, 10000])
tensor([[3959],
        [7970]])


In [ ]:
# LangChain
%pip install langchain
%pip install langchain-core
%pip install langchain-community
%pip install langchain-openai

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os
from dotenv import load_dotenv

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_API_URL = os.getenv("OPENAI_API_URL")

prompt = ChatPromptTemplate.from_messages([
    ("system", "{system_message}"),
    ("human", "{question}")
])
llm = ChatOpenAI(
    model = "gpt-4o",
    api_key=OPENAI_API_KEY
)
chain = prompt | llm | StrOutputParser()
response = chain.invoke({
    "system_message": "You are a senior LangChain programmer",
    "question": "What is the difference between OpenAI SDK and LangChain?",
})
print(response)


In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)
prompt = "Explain what LangChain is in three sentences."
response = client.responses.create(
    model="gpt-4o",
    input=prompt,
)
print(response.output_text)